In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# paths
DATA = r'C:\Users\shahb\Desktop\Research\Datasets'
EMR = DATA + r'\EPIC EMR\EPIC_EMR\EMR'
MOVER = DATA + r'\MOVER'

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

In [ ]:
# load pt info
pt = pd.read_csv(EMR + r'\patient_information.csv')
print(pt.shape, pt.columns.tolist())
pt.head(3)

In [ ]:
# exclude cardiac surgery - cabg, valves etc
excl = pt['PRIMARY_PROCEDURE_NM'].str.lower().str.contains(
    'cabg|bypass graft|valve|aortic repair|cardiopulmonary', na=False)
print('cardiac surg excluded:', excl.sum())
pt2 = pt[~excl].copy()

In [ ]:
# load complications - need to figure out outcome
cx = pd.read_csv(EMR + r'\patient_post_op_complications.csv')
print(cx['SMRTDTA_ELEM_VALUE'].value_counts(dropna=False).head(20))

In [ ]:
# cardiac outcome - tried broad first then realized strict gives same result
cardiac_kw = ['Cardiovascular', 'Arrest (cardiac)', 'Ischemia (myocardial)',
              'Arrhythmia', 'Instability (hemodynamic)']
cardiac_ids = cx[cx['SMRTDTA_ELEM_VALUE'].isin(cardiac_kw)]['LOG_ID'].unique()
print(len(cardiac_ids), 'unique log ids w cardiac cx')

# sanity check - strict def
strict_kw = ['Cardiovascular', 'Arrest (cardiac)', 'Ischemia (myocardial)']
strict_ids = cx[cx['SMRTDTA_ELEM_VALUE'].isin(strict_kw)]['LOG_ID'].unique()
print(len(strict_ids), 'strict - same??')  # turned out identical

In [ ]:
# load flowsheets - already preprocessed
fs = pd.read_csv(MOVER + r'\flowsheets_master.csv', dtype={'MEAS_VALUE': str})
print(fs.shape)
fs['MEAS_VALUE'] = pd.to_numeric(fs['MEAS_VALUE'], errors='coerce')

In [ ]:
# hr features per surgery
hr = fs[fs['FLO_DISPLAY_NAME'] == 'Heart Rate']
hr_feats = hr.groupby('LOG_ID')['MEAS_VALUE'].agg(
    HR_mean='mean', HR_min='min', HR_max='max', HR_std='std').reset_index()
print(hr_feats.shape)
hr_feats.describe()

In [ ]:
# spo2
spo2 = fs[fs['FLO_DISPLAY_NAME'] == 'SpO2']
spo2_feats = spo2.groupby('LOG_ID')['MEAS_VALUE'].agg(
    SpO2_mean='mean', SpO2_min='min').reset_index()

# map - sparse, only art line cases
mapdf = fs[fs['FLO_DISPLAY_NAME'].isin(['MAP (mmHg)', 'Arterial Line MAP (ART)'])]
mapdf = mapdf.copy()
mapdf['hypo'] = (mapdf['MEAS_VALUE'] < 65).astype(int)
map_feats = mapdf.groupby('LOG_ID')['MEAS_VALUE'].agg(
    MAP_mean='mean', MAP_min='min', MAP_std='std').reset_index()
map_hypo = mapdf.groupby('LOG_ID')['hypo'].mean().reset_index().rename(
    columns={'hypo': 'MAP_pct_below65'})
map_feats = map_feats.merge(map_hypo, on='LOG_ID')
print('map cases:', len(map_feats))  # only ~4k, expected

In [ ]:
# merge intraop
intraop = hr_feats.merge(spo2_feats, on='LOG_ID', how='outer')
intraop = intraop.merge(map_feats, on='LOG_ID', how='outer')

In [ ]:
# comorbidities from hx
hx = pd.read_csv(EMR + r'\patient_history.csv')

def flag_dx(mrns, codes):
    m = hx['diagnosis_code'].str.startswith(tuple(codes), na=False)
    return set(hx[m]['mrn'])

# standard periop comorbidities
htn_mrns = flag_dx(None, ['401', 'I10', 'I11', 'I12', 'I13'])
dm_mrns  = flag_dx(None, ['250', 'E11', 'E10', 'E12', 'E13'])
chf_mrns = flag_dx(None, ['428', 'I50'])
cad_mrns = flag_dx(None, ['414', 'I25'])
copd_mrns = flag_dx(None, ['496', 'J44'])
ckd_mrns = flag_dx(None, ['585', 'N18'])
mi_mrns  = flag_dx(None, ['410', 'I21', 'I22'])
afib_mrns = flag_dx(None, ['427.31', 'I48'])
stroke_mrns = flag_dx(None, ['434', 'I63', 'I64'])
pvd_mrns = flag_dx(None, ['443', 'I73'])

In [ ]:
# build analytic dataset
df = pt2[pt2['LOG_ID'].isin(fs['LOG_ID'].unique())].copy()
df['SEX_encoded'] = (df['SEX'] == 'Male').astype(int)
df['AGE'] = pd.to_numeric(df['BIRTH_DATE'], errors='coerce')
df['IN_OR_DTTM'] = pd.to_datetime(df['IN_OR_DTTM'], infer_datetime_format=True)
df['OUT_OR_DTTM'] = pd.to_datetime(df['OUT_OR_DTTM'], infer_datetime_format=True)
df['OPERATIVE_DURATION'] = (df['OUT_OR_DTTM'] - df['IN_OR_DTTM']).dt.total_seconds()/60

# outcome
df['CARDIAC_OUTCOME'] = df['LOG_ID'].isin(cardiac_ids).astype(int)

# comorbidities
df['hypertension'] = df['MRN'].isin(htn_mrns).astype(int)
df['diabetes']     = df['MRN'].isin(dm_mrns).astype(int)
df['chf']          = df['MRN'].isin(chf_mrns).astype(int)
df['cad']          = df['MRN'].isin(cad_mrns).astype(int)
df['copd']         = df['MRN'].isin(copd_mrns).astype(int)
df['ckd']          = df['MRN'].isin(ckd_mrns).astype(int)
df['prior_mi']     = df['MRN'].isin(mi_mrns).astype(int)
df['afib']         = df['MRN'].isin(afib_mrns).astype(int)
df['stroke']       = df['MRN'].isin(stroke_mrns).astype(int)
df['pvd']          = df['MRN'].isin(pvd_mrns).astype(int)

# merge intraop
df = df.merge(intraop, on='LOG_ID', how='left')
print(df.shape, df['CARDIAC_OUTCOME'].sum())

In [ ]:
# quick check on missingness before modeling
miss = df[['ASA_RATING_C','HR_mean','SpO2_mean','MAP_mean']].isna().mean()*100
print(miss.round(1))
# MAP is very sparse as expected - only art line cases

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import shap

feats_preop = ['AGE','SEX_encoded','ASA_RATING_C','OPERATIVE_DURATION',
               'hypertension','diabetes','chf','cad','copd',
               'ckd','prior_mi','afib','stroke','pvd']

feats_m2 = feats_preop + ['HR_mean','HR_min','HR_max','HR_std','SpO2_mean','SpO2_min']

y = df['CARDIAC_OUTCOME']

X1 = df[feats_preop].copy()
X1['ASA_RATING_C'] = X1['ASA_RATING_C'].fillna(X1['ASA_RATING_C'].median())

X2 = df[feats_m2].copy()
X2['ASA_RATING_C'] = X2['ASA_RATING_C'].fillna(X2['ASA_RATING_C'].median())

X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y, test_size=.2, 
                                                 random_state=42, stratify=y)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=.2,
                                                 random_state=42, stratify=y)

In [ ]:
# model 1 - LR baseline
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X1_tr, y1_tr)
lr_proba = lr.predict_proba(X1_te)[:,1]
print('LR AUC:', roc_auc_score(y1_te, lr_proba).round(3))

In [ ]:
# model 2 - xgb with intraop
# scale_pos_weight handles imbalance
spw = len(y2_tr)/y2_tr.sum()
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=.05,
                    scale_pos_weight=spw, random_state=42, eval_metric='auc')
xgb.fit(X2_tr, y2_tr)
xgb_proba = xgb.predict_proba(X2_te)[:,1]
print('XGB AUC:', roc_auc_score(y2_te, xgb_proba).round(3))
print('XGB AUC-PR:', average_precision_score(y2_te, xgb_proba).round(3))

In [ ]:
# rcri - compute on same test set
rcri_kw = ['laparotomy','thoracot','aortic','vascular','colectomy',
           'gastrectomy','esophag','hepat','pancreat']
df['high_risk_surg'] = df['PRIMARY_PROCEDURE_NM'].str.lower().str.contains(
    '|'.join(rcri_kw), na=False).astype(int)
df['cr_high'] = (df['Creatinine'] > 2.0).astype(int).fillna(0)
df['RCRI'] = (df['high_risk_surg'] + df['prior_mi'] + df['cad'] +
              df['chf'] + df['stroke'] + df['cr_high'] + df['diabetes'])

rcri_te = df.loc[X2_te.index, 'RCRI']
print('RCRI AUC:', roc_auc_score(y2_te, rcri_te).round(3))

In [ ]:
# bootstrap CIs - slow but worth it
def boot_auc(y_true, y_pred, n=1000):
    aucs = []
    for _ in range(n):
        idx = np.random.randint(0, len(y_true), len(y_true))
        if y_true.iloc[idx].sum() == 0: continue
        aucs.append(roc_auc_score(y_true.iloc[idx], y_pred[idx]))
    return np.percentile(aucs, [2.5, 97.5])

ci_rcri = boot_auc(y2_te, rcri_te.values)
ci_lr   = boot_auc(y1_te, lr_proba)
ci_xgb  = boot_auc(y2_te, xgb_proba)

print(f'RCRI  {roc_auc_score(y2_te, rcri_te):.3f} ({ci_rcri[0]:.3f}-{ci_rcri[1]:.3f})')
print(f'LR    {roc_auc_score(y1_te, lr_proba):.3f} ({ci_lr[0]:.3f}-{ci_lr[1]:.3f})')
print(f'XGB   {roc_auc_score(y2_te, xgb_proba):.3f} ({ci_xgb[0]:.3f}-{ci_xgb[1]:.3f})')

In [ ]:
# shap - main interpretability
explainer = shap.TreeExplainer(xgb)
shap_vals = explainer.shap_values(X2_te)

shap.summary_plot(shap_vals, X2_te, plot_type='bar', show=False)
plt.title('Feature importance (SHAP)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure3_shap_bar_mover.png', dpi=300)
plt.show()

In [ ]:
shap.summary_plot(shap_vals, X2_te, show=False)
plt.title('SHAP beeswarm - MOVER', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure4_shap_beeswarm_mover.png', dpi=300)
plt.show()

In [ ]:
# calibration + platt scaling
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

cal_xgb = CalibratedClassifierCV(xgb, method='sigmoid', cv='prefit')
cal_xgb.fit(X2_tr, y2_tr)
cal_proba = cal_xgb.predict_proba(X2_te)[:,1]

fig, ax = plt.subplots(figsize=(7,6))
ax.plot([0,1],[0,1],'k--',lw=1,label='Perfect')
for proba, label in [(lr_proba,'LR preop'), (xgb_proba,'XGB uncal'), (cal_proba,'XGB cal')]:
    frac, mean = calibration_curve(y2_te if 'LR' not in label else y1_te, 
                                    proba, n_bins=10)
    ax.plot(mean, frac, 's-', lw=2, label=label)
ax.set(xlabel='Mean predicted prob', ylabel='Fraction positives',
       title='Calibration curves')
ax.legend()
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure2_calibration.png', dpi=300)
plt.show()

In [ ]:
# roc curves - all models
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7,6))
for proba, y_true, label, ls, c in [
    (rcri_te.values, y2_te, f'RCRI (AUC=0.671)', ':', 'red'),
    (lr_proba, y1_te, f'Model 1 LR (AUC=0.753)', '-', 'blue'),
    (xgb_proba, y2_te, f'Model 2 XGB (AUC=0.823)', '-', 'orange'),
]:
    fpr, tpr, _ = roc_curve(y_true, proba)
    ax.plot(fpr, tpr, ls=ls, color=c, lw=2, label=label)

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves')
ax.legend(fontsize=10)
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure1_roc_curves.png', dpi=300)
plt.show()